# Set up a Snowflake Managed MCP Server for a Gemini Agent

This notebook **stands up and validates the MCP tool surface** that a Google Cloud **Gemini agent** connects to. We build the objects into tools, add a read-only SQL tool so the agent returns real numbers, verify the server is live, and issue the token the agent authenticates with. The **live investigation happens in the Gemini agent** — this notebook is the setup cockpit behind it.

**The use case:** a CX analyst, working in their Gemini agent, investigates a customer escalation — searches reviews, quantifies the sales impact, and opens a **refund-approval case** that a **human** signs off on before any money moves.

**Why MCP and not a scheduled job?** A nightly "flag bad SKUs" rule *should* be a Snowflake Task. MCP earns its place when a **human asks unpredictable questions in the agent they already live in**, and that agent composes your governed Snowflake tools on the fly. Schedule the predictable; expose over MCP the ad hoc, human-in-the-loop work.

## The objects this notebook wires into tools

| Object | What it is | Exposed to Gemini as |
|---|---|---|
| `MCP_HOL.SUPPORT.CLASSIFY_INTENT` | UDF over **SUPPORT_INTENT_8B** (Llama 3.1 8B fine-tuned in-account) | `classify_intent` |
| `MCP_HOL.SUPPORT.SEARCH_REVIEWS` | Cortex Search service over customer reviews | `search_reviews` |
| `MCP_HOL.SALES.SALES_SV` | semantic view for Cortex Analyst | `ask_sales_data` |
| *(read-only SQL on the account)* | SELECT-only execution to quantify impact | `run_sql` |
| `MCP_HOL.SUPPORT.CREATE_TICKET` | procedure: opens a case (Zendesk-ready), **does not refund** | `create_support_ticket` |
| `MCP_HOL.SUPPORT.APPROVE_CASE` | procedure: the **human** approves → refund released | **not exposed** |
| `MCP_HOL.AGENTS.MCP_HOL` | the MCP server itself | — |

Supporting data: `MCP_HOL.SUPPORT.REVIEWS`, `MCP_HOL.SALES.SALES_FACT`, `MCP_HOL.SUPPORT.TICKETS` (the case log), and `MCP_HOL.SUPPORT.INTENT_TRAIN` / `INTENT_PROBE` (the fine-tune data).

## Section 1 — The data behind the tools

Raw reviews for the product under investigation, then a Cortex Search preview (testing-only — Gemini hits this semantically via the `search_reviews` tool).

In [ ]:
SELECT PRODUCT, RATING, REVIEW_TEXT, REVIEW_DATE
FROM MCP_HOL.SUPPORT.REVIEWS
WHERE PRODUCT = 'Summit Winter Jacket'
ORDER BY REVIEW_DATE DESC
LIMIT 15

In [ ]:
-- Testing-only preview. The agent calls this semantically via the search_reviews MCP tool.
SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'MCP_HOL.SUPPORT.SEARCH_REVIEWS',
  '{"query":"zipper keeps breaking","columns":["REVIEW_TEXT","PRODUCT","RATING"],"limit":5}'
) AS search_results

## Section 2 — The fine-tuned classifier tool (own the decision boundary)

`classify_intent` is the one tool whose **decision boundary you own**. It wraps **`SUPPORT_INTENT_8B`** — a **Llama 3.1 8B fine-tuned in this account** on our own labeled support messages — sorting each message into one of six intents: `ORDER_STATUS`, `SHIPPING_DELAY`, `DEFECTIVE_ITEM`, `RETURN_REFUND`, `SIZING_EXCHANGE`, `GENERAL_FEEDBACK`. A frontier model can be *prompted*; only a fine-tune is *trained on your labels*.

**How it was built** — run ONCE, outside this notebook (fine-tuning is long-running; Run All must stay fast):
```sql
SELECT SNOWFLAKE.CORTEX.FINETUNE('CREATE',
  'MCP_HOL.SUPPORT.SUPPORT_INTENT_8B', 'llama3.1-8b',
  $$SELECT <instruction> || MESSAGE || ' Label:' AS prompt, INTENT AS completion FROM MCP_HOL.SUPPORT.INTENT_TRAIN$$,
  $$ ...same instruction over INTENT_VAL$$ );
```
~120 labeled rows, a few minutes, pennies. **Note:** fine-tune jobs are garbage-collected over weeks — recreate close to the event and re-verify (full script: `demo/06_finetune_intent.sql`). The cells below just **call** the trained model.

In [ ]:
-- The GENERIC tool calls this UDF, which runs the fine-tuned model.
SELECT column1 AS message, MCP_HOL.SUPPORT.CLASSIFY_INTENT(column1) AS predicted_intent
FROM VALUES
  ('The zipper broke the first time I wore it'),
  ('still not here?? ordered 3 wks ago'),
  ('can I swap this for a larger size?'),
  ('money back pls'),
  ('10/10 coat, love it')

**Does the fine-tune earn its place?** Score it against base `llama3.1-8b` zero-shot on a held-out probe (clean + messy real-world messages):

In [ ]:
WITH scored AS (
  SELECT INTENT AS actual,
    MCP_HOL.SUPPORT.CLASSIFY_INTENT(MESSAGE) AS ft_pred,
    UPPER(TRIM(SNOWFLAKE.CORTEX.COMPLETE('llama3.1-8b',
      'Classify the customer support message into exactly one label from this list: ORDER_STATUS, SHIPPING_DELAY, DEFECTIVE_ITEM, RETURN_REFUND, SIZING_EXCHANGE, GENERAL_FEEDBACK. Reply with only the label and nothing else. Message: ' || MESSAGE || ' Label:'))) AS base_pred
  FROM MCP_HOL.SUPPORT.INTENT_PROBE
)
SELECT COUNT(*) AS probe_rows,
  ROUND(AVG(IFF(ft_pred=actual,1,0))*100,1) AS ft_accuracy_pct,
  ROUND(AVG(IFF(base_pred=actual,1,0))*100,1) AS base_zeroshot_accuracy_pct
FROM scored

## Section 3 — Sales impact (what `run_sql` will quantify)

Refunds by product. The Summit Winter Jacket tops the list (~320 units, ~$9K refunds already). This is the kind of figure the Gemini agent returns live by calling `run_sql`.

In [ ]:
SELECT PRODUCT, SUM(UNITS) AS units, SUM(REFUND_AMT) AS refunds
FROM MCP_HOL.SALES.SALES_FACT
GROUP BY 1
ORDER BY refunds DESC

## Section 4 — Build the tool surface (run this)

One `CREATE MCP SERVER` statement turns Snowflake objects into agent tools. Two design choices make Gemini interact well **and** stay governed:

- **`run_sql` (`SYSTEM_EXECUTE_SQL`, `read_only: true`)** — `ask_sales_data` (Cortex Analyst) returns the *generated SQL* but not executed rows, so numbers don't surface in chat. A read-only SQL tool lets Gemini **execute** and return the actual figures — SELECT only, no writes.
- **`APPROVE_CASE` is deliberately NOT a tool** — the agent can *open* a refund-approval case; only a human can *approve* it. That boundary is the tool surface, not a prompt.

Running this cell **(re)deploys the live server** the Gemini agent connects to.

In [ ]:
CREATE OR REPLACE MCP SERVER MCP_HOL.AGENTS.MCP_HOL
FROM SPECIFICATION $$
tools:
  - name: "classify_intent"   # fine-tuned Llama 3.1 8B, wrapped as a GENERIC tool
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.CLASSIFY_INTENT"
    title: "Classify a support message (fine-tuned model)"
    description: "Classify a customer-support message into one intent: ORDER_STATUS, SHIPPING_DELAY, DEFECTIVE_ITEM, RETURN_REFUND, SIZING_EXCHANGE, GENERAL_FEEDBACK. Backed by SUPPORT_INTENT_8B, fine-tuned in-account on our labels."
    config:
      type: "function"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          message: { type: "string", description: "The customer's support message text" }
        required: ["message"]
  - name: "search_reviews"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "MCP_HOL.SUPPORT.SEARCH_REVIEWS"
    title: "Search customer reviews"
    description: "Search customer reviews for a topic and return the most relevant reviews (shipping, sizing, zipper problems, etc.)."
  - name: "ask_sales_data"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "MCP_HOL.SALES.SALES_SV"
    title: "Ask sales data"
    description: "Ask natural-language questions about sales: units, revenue, and refunds by product, region, or date. Returns the interpretation and generated SQL."
  - name: "run_sql"
    type: "SYSTEM_EXECUTE_SQL"
    title: "Run read-only SQL"
    description: "Run a read-only SQL SELECT to quantify impact - e.g. count affected orders or sum refunds. Discover tables/columns via INFORMATION_SCHEMA if needed. Use this to return the actual numbers."
    config:
      read_only: true
      warehouse: "COCO_WH"
  - name: "create_support_ticket"   # opens a refund-approval case; does NOT refund
    type: "GENERIC"
    identifier: "MCP_HOL.SUPPORT.CREATE_TICKET"
    title: "Open a refund-approval case"
    description: "Open a refund-approval CASE for an order. Does NOT issue a refund - records the recommendation and routes it to a human. Provide order_id, a short issue, and the recommended refund amount."
    config:
      type: "procedure"
      warehouse: "COCO_WH"
      input_schema:
        type: "object"
        properties:
          order_id: { type: "string", description: "The affected order id, e.g. ORD-10001" }
          issue: { type: "string", description: "Short description of the customer problem" }
          recommended_refund: { type: "number", description: "Recommended refund amount in dollars" }
        required: ["order_id", "issue", "recommended_refund"]
$$;

## Section 5 — Verify the server and its tools

Confirm the server exists and see the five tools Gemini will discover via `tools/list`.

In [ ]:
SHOW MCP SERVERS IN SCHEMA MCP_HOL.AGENTS;
-- Then inspect the tool spec:
-- DESCRIBE MCP SERVER MCP_HOL.AGENTS.MCP_HOL;

## Section 6 — Connect the Gemini agent (Google Cloud)

The Gemini agent authenticates to the managed MCP server with a **programmatic access token (PAT)**. The token's role restriction **must match your `DEFAULT_ROLE`** — the managed server runs every call under that role (no secondary roles), on your default warehouse.

Run the cell below, copy `token_secret` **once**, and set it as `SNOWFLAKE_PAT` in the Gemini agent's `.env`.

**MCP endpoint** (note the host uses **hyphens**):
```
https://sfsenorthamerica-zhada-aws1.snowflakecomputing.com/api/v2/databases/MCP_HOL/schemas/AGENTS/mcp-servers/MCP_HOL
```

**Google ADK wiring** (in the agent, on Google Cloud — streamable-HTTP transport):
```python
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

MCP_URL = "https://sfsenorthamerica-zhada-aws1.snowflakecomputing.com/api/v2/databases/MCP_HOL/schemas/AGENTS/mcp-servers/MCP_HOL"

agent = LlmAgent(
    model="gemini-3.1-pro-preview",   # served only from the 'global' Vertex location
    name="snowflake_mcp_agent",
    instruction="You are a CX analyst. Use the Snowflake tools to investigate escalations, "
                "quantify impact with run_sql, and open a refund-approval case when a defect is real.",
    tools=[MCPToolset(connection_params=StreamableHTTPConnectionParams(
        url=MCP_URL,
        headers={"Authorization": f"Bearer {SNOWFLAKE_PAT}"},
    ))],
)
```
ADK runs `tools/list` at startup; Gemini then calls the tools via `tools/call`.

In [ ]:
-- Issue the token the Gemini agent authenticates with. ROLE_RESTRICTION must equal your DEFAULT_ROLE.
-- Copy token_secret ONCE -> Gemini agent .env as SNOWFLAKE_PAT.
ALTER USER ZHADA ADD PROGRAMMATIC ACCESS TOKEN gemini_mcp_pat
  ROLE_RESTRICTION = 'ACCOUNTADMIN'
  DAYS_TO_EXPIRY = 7;

## Section 7 — Baseline: open cases BEFORE the agent runs

In [ ]:
SELECT TICKET_ID, ORDER_ID, RECOMMENDED_REFUND, STATUS, EXTERNAL_CASE, APPROVED_BY
FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

## Section 8 — RUN THE AGENT (in the Gemini agent, on Google Cloud)

Switch to the **Gemini agent** (`adk web`, or your deployed Agent Engine endpoint) and paste an ad-hoc request — the kind you'd never pre-script:

---

> *A customer emailed that the zipper on their Summit Winter Jacket (order ORD-10001) broke. Is this a known problem, and how big is it? If it's a real defect, open a refund-approval case for their order with a recommended refund.*

---

**Watch Gemini compose the tools you just built:**

1. `classify_intent` — the fine-tuned model triages the email as **DEFECTIVE_ITEM**
2. `search_reviews` — finds the zipper-defect theme across the reviews
3. `ask_sales_data` → `run_sql` — interprets the ask over the semantic view, then **executes** to return the real numbers (~320 units, ~$9K refunds)
4. `create_support_ticket` — opens a **PENDING_APPROVAL** case recommending the refund

The agent stops there. It has no tool to approve or move money — that boundary is enforced by what you exposed, not by a prompt.

## Section 9 — The case the agent opened (PENDING approval)

Re-run: a new case appears, `STATUS = PENDING_APPROVAL`, with the agent's recommended refund and (when reachable) the Zendesk case id.

In [ ]:
SELECT TICKET_ID, ORDER_ID, RECOMMENDED_REFUND, STATUS, EXTERNAL_CASE
FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

## Section 10 — The human approves (the step the agent can't do)

A CX manager reviews the case and approves it. Approval releases the refund downstream — a privileged action deliberately kept out of the agent's tool surface. Uses the most recent case from Section 9.

In [ ]:
CALL MCP_HOL.SUPPORT.APPROVE_CASE(
  (SELECT TICKET_ID FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC LIMIT 1)
)

In [ ]:
SELECT TICKET_ID, ORDER_ID, RECOMMENDED_REFUND, STATUS, APPROVED_BY, APPROVED_AT
FROM MCP_HOL.SUPPORT.TICKETS ORDER BY CREATED_AT DESC

## Recap

- **The notebook is the setup, the agent is the show** — you built and validated a governed tool surface here; the live, unscripted investigation runs in the Gemini agent against it.
- **Tuned for the client** — adding a read-only `run_sql` tool closed the Cortex Analyst "SQL but no rows" gap, so Gemini returns actual figures.
- **Build once, any client, your RBAC** — one `CREATE MCP SERVER` exposed five tools; every call runs under the connecting user's role.
- **Own the decision boundary** — `classify_intent` wraps a Llama 3.1 8B you fine-tuned in-account (100% vs 97.2% base zero-shot on a held-out probe); a frontier model can be prompted, only a fine-tune learns your labels.
- **The agent acts, but within a boundary** — it opens a governed case and recommends a refund; only a human approves. That boundary is the tool surface you chose to expose.

## Appendix A — The five MCP tool types

One MCP server can expose up to 50 tools. Every call runs under the **connecting user's DEFAULT_ROLE** (no secondary roles) on their DEFAULT_WAREHOUSE.

| Type | What it exposes | Used here |
|---|---|---|
| `CORTEX_SEARCH_SERVICE_QUERY` | Semantic search over a Cortex Search service | `search_reviews` |
| `CORTEX_ANALYST_MESSAGE` | Natural-language → SQL over a semantic view | `ask_sales_data` |
| `SYSTEM_EXECUTE_SQL` | Run SQL directly (read-only here) | `run_sql` |
| `GENERIC` | Wrap one UDF/procedure as a typed action | `create_support_ticket`, `classify_intent` |
| `CORTEX_AGENT_RUN` | Expose an entire Cortex Agent as one tool | — (see Appendix C) |

**`SYSTEM_EXECUTE_SQL` config** — powerful, so scope it deliberately:
- `read_only`: `true` = SELECT only (default) · `false` = writes + DDL
- `query_timeout`: max seconds per query
- `warehouse`: compute to run on (else the user's default)

**`GENERIC` config** — the safest 'action' tool; one named operation, typed inputs, no freehand SQL:
- `identifier`: the UDF or procedure to wrap
- `type`: `function` (UDF) or `procedure`
- `input_schema`: the typed args the agent must fill
- `warehouse` / `query_timeout`: compute + limit

## Appendix B — How `create_support_ticket` reaches Zendesk (External Access)

The GENERIC tool wraps a Python procedure that makes a governed outbound call. The reach-out is built from a network rule + secret + external access integration — the agent never sees the credentials.

```sql
-- 1) allow egress to the CX host
CREATE NETWORK RULE ZENDESK_RULE MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('acme.zendesk.com');
-- 2) store the credential
CREATE SECRET ZENDESK_CRED TYPE=PASSWORD USERNAME='agent@acme.com/token' PASSWORD='<api_token>';
-- 3) bind them into an integration
CREATE EXTERNAL ACCESS INTEGRATION ZENDESK_MCP_HOL_INT
  ALLOWED_NETWORK_RULES=(ZENDESK_RULE) ALLOWED_AUTHENTICATION_SECRETS=(ZENDESK_CRED) ENABLED=TRUE;

-- 4) the procedure the tool wraps: opens a PENDING case, does NOT refund
CREATE PROCEDURE SUPPORT.CREATE_TICKET(order_id STRING, issue STRING, recommended_refund FLOAT)
  RETURNS STRING LANGUAGE PYTHON RUNTIME_VERSION=3.11 HANDLER='run'
  EXTERNAL_ACCESS_INTEGRATIONS=(ZENDESK_MCP_HOL_INT)
  PACKAGES=('snowflake-snowpark-python','requests') SECRETS=('cred'=ZENDESK_CRED)
AS $$
import _snowflake, requests
def run(session, order_id, issue, recommended_refund):
    c = _snowflake.get_username_password('cred')
    r = requests.post('https://acme.zendesk.com/api/v2/tickets.json',
        auth=(c.username, c.password),
        json={'ticket': {'subject': 'Refund approval: order ' + str(order_id),
                         'comment': {'body': issue + ' | $' + str(recommended_refund) + ' | PENDING_APPROVAL'}}})
    # ...insert a PENDING_APPROVAL row into SUPPORT.TICKETS with the Zendesk case id...
$$;
```

The matching `APPROVE_CASE(ticket_id)` procedure — the human step that releases the refund — is **not** exposed over MCP.

## Appendix C — Other ways to expose over MCP

You are not limited to hand-picked tools. Two shortcuts:

**Expose an entire Cortex Agent** (runs in Snowflake):
```sql
CREATE MCP SERVER support_api FROM SPECIFICATION $$
tools:
  - name: "support_agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "db.agents.support_agent"
$$;
```

**Expose Cortex Code (CoCo) itself** from your machine:
```bash
cortex mcp serve -c my_conn --bypass
```
...which hands the client `cortex_code_agent`, `cortex_analyst_query`, `cortex_search_objects`, plus `agents_list` / `search_docs`.